# Synthetic Nanotube Forensics — `stage0_survivors` set

Structural forensics of the **synthetic** stage-0 survivor tubes
(`stage0_survivors_structures.pkl`, 31,866 raw pre-relaxation ASE structures),
the synthetic half of the `shl` pinning-template database.

This mirrors `nanotube_rtheta_forensics.ipynb` (which analyses the *real*
Alexandria set) but adds an **axial reduction to ≤128 atoms** — the same shrink
`build_templates.py` applies before these tubes become pinning cells — so every
metric reflects the geometry the mask actually sees.

> These are **raw generated candidates** (pre-relaxation): expect unphysically
> short contacts. The quality filter at the end quantifies how many are usable.
> **Nothing is written to disk.**

In [ ]:
# --- dependencies (run once) -------------------------------------------------
%pip install -q numpy pandas matplotlib scipy scikit-learn
import numpy, pandas, matplotlib, scipy, sklearn
for m in (numpy, pandas, matplotlib, scipy, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
import sys, json, math, time, warnings, pickle
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.signal import find_peaks
from scipy.spatial import cKDTree
from sklearn.mixture import GaussianMixture

# ---- configuration ----------------------------------------------------------
PKL_PATH  = Path("/Users/evansmacbookair/Downloads/NanotubeData/"
                 "stage0_survivors_structures.pkl.download/stage0_survivors_structures.pkl")
REDUCE_DIR = Path("/Users/evansmacbookair/GitHub/NTU-IQM/NTGENS/data/nano_1D")  # ase-optional loaders
REDUCE_TARGET = 128      # axial-reduce every tube to <= this many atoms (DB cap)
MAX_LOAD  = None         # None = all 31,866 (~5-8 min); set an int for a fast subset
SEED      = 0
N_SAMPLE  = 25           # tubes for detailed per-tube pass (unused-heavy cells)

rng = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")

# ---- plot style (matches the real-set notebook) -----------------------------
ACCENT = ["#43AA8B", "#F8961E", "#F94144", "#277DA1"]
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
mpl.rcParams.update({
    "font.size": 12, "figure.dpi": 110, "figure.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
})

def elem_colors(elements):
    "Stable element -> colour map (CVD-safe, alphabetical) for a single structure."
    uniq = sorted(set(map(str, elements)))
    return {el: OKABE_ITO[i % len(OKABE_ITO)] for i, el in enumerate(uniq)}

# Z -> symbol, no ase/pymatgen (index = atomic number; X=0).
_SYMBOLS = ['X','H','He','Li','Be','B','C','N','O','F','Ne','Na','Mg','Al','Si','P','S','Cl',
    'Ar','K','Ca','Sc','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn','Ga','Ge','As','Se','Br',
    'Kr','Rb','Sr','Y','Zr','Nb','Mo','Tc','Ru','Rh','Pd','Ag','Cd','In','Sn','Sb','Te','I',
    'Xe','Cs','Ba','La','Ce','Pr','Nd','Pm','Sm','Eu','Gd','Tb','Dy','Ho','Er','Tm','Yb','Lu',
    'Hf','Ta','W','Re','Os','Ir','Pt','Au','Hg','Tl','Pb','Bi','Po','At','Rn','Fr','Ra','Ac',
    'Th','Pa','U','Np','Pu','Am','Cm','Bk','Cf','Es','Fm','Md','No','Lr']

def symbols_of(numbers):
    return np.array([_SYMBOLS[z] if 0 <= z < len(_SYMBOLS) else str(z) for z in numbers])

def formula_of(numbers):
    c = Counter(symbols_of(numbers))
    return "".join(f"{el}{c[el] if c[el] > 1 else ''}" for el in sorted(c))

print("Config ready. Pickle exists:", PKL_PATH.exists())

## 1 · Load & axially reduce the synthetic tubes

The pickle is a list of ASE `Atoms`; `ase` is not installed so we unpickle it via
the repo's **ase-optional stubs** (`build_templates._load_ase_pickle`) that keep
only numbers/positions/cell. Each tube is then folded to a ≤128-atom repeat unit
with `reduce_templates.reduce_structure` — exactly what the template builder does.

In [ ]:
# import the proven ase-optional loaders + tiered reducer (numpy/scipy only)
if str(REDUCE_DIR) not in sys.path:
    sys.path.insert(0, str(REDUCE_DIR))
from build_templates import _load_ase_pickle, _numbers_positions_cell
from reduce_templates import reduce_structure

t0 = time.time()
db = _load_ase_pickle(str(PKL_PATH))
n_total = len(db)
idxs = range(n_total if MAX_LOAD is None else min(MAX_LOAD, n_total))
print(f"Unpickled {n_total:,} structures in {time.time() - t0:.1f}s; "
      f"processing {len(idxs):,} (reduce -> <= {REDUCE_TARGET} atoms)...")

templates, skipped = [], 0
t0 = time.time()
for i in idxs:
    nums, pos, cell = _numbers_positions_cell(db[i])
    cell = np.asarray(cell, float); pos = np.asarray(pos, float)
    if not np.isfinite(np.linalg.det(cell)) or abs(np.linalg.det(cell)) < 1e-6:
        skipped += 1; continue                      # degenerate cell
    n_raw = len(nums)
    frac = pos @ np.linalg.inv(cell); frac -= np.floor(frac)
    red = reduce_structure(np.asarray(nums), frac, cell, target=REDUCE_TARGET)
    if red is None:
        skipped += 1; continue                      # Tier-3 drop (un-reducible tube)
    rn, rf, rc, tag = red
    rf = rf - np.floor(rf)
    templates.append(dict(
        mat_id=None, formula=formula_of(rn), nsites=len(rn), nsites_raw=n_raw,
        reduce_tag=tag, e_above_hull=None,
        cell=np.asarray(rc, float), frac=np.asarray(rf, float),
        cart=np.asarray(rf, float) @ np.asarray(rc, float),
        numbers=np.asarray(rn), elements=symbols_of(rn),
    ))
del db  # free the big list
print(f"Built {len(templates):,} template records in {time.time() - t0:.1f}s "
      f"({skipped} skipped: degenerate cell or un-reducible).")

_nr = np.array([t["nsites_raw"] for t in templates])
_nd = np.array([t["nsites"] for t in templates])
print(f"\natoms/cell   raw: median {np.median(_nr):.0f}, max {_nr.max()}   "
      f"reduced: median {np.median(_nd):.0f}, max {_nd.max()}")
print("reduce tags:", dict(Counter(t["reduce_tag"] for t in templates)))

## 2 · Geometry helpers (shared with the real-set notebook)

Identical tube-axis / cylindrical-coordinate / radial-density / bonding primitives
as `nanotube_rtheta_forensics.ipynb`, so metrics are directly comparable.

In [ ]:
def detect_tube_axis(frac):
    "Lattice direction the atoms fill most (smallest circular vacuum gap)."
    occ = []
    for k in range(3):
        f = np.sort(frac[:, k] % 1.0)
        if len(f) < 2:
            occ.append(0.0); continue
        gaps = np.diff(np.concatenate([f, [f[0] + 1.0]]))
        occ.append(1.0 - gaps.max())
    return int(np.argmax(occ))

def cylindrical_coords(cart, cell, axis):
    "Return r, theta, z_axial, u, v in the frame perpendicular to the tube axis."
    a_hat = cell[axis] / np.linalg.norm(cell[axis])
    z = cart @ a_hat
    perp = cart - np.outer(z, a_hat)
    other = [k for k in range(3) if k != axis]
    e1 = cell[other[0]] - (cell[other[0]] @ a_hat) * a_hat
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(a_hat, e1)
    ctr = perp.mean(0)
    u = (perp - ctr) @ e1
    v = (perp - ctr) @ e2
    return np.hypot(u, v), np.arctan2(v, u), z, u, v

def radial_density(r, L, nbins=30):
    "Jacobian-corrected radial number density rho(r)."
    if r.size < 2 or np.ptp(r) < 1e-6:
        return np.array([float(r.mean()) if r.size else 0.0]), \
               np.array([float(r.size)]), np.array([r.size])
    nb = int(min(nbins, max(3, r.size)))
    counts, edges = np.histogram(r, bins=nb)
    ctr = 0.5 * (edges[:-1] + edges[1:]); dr = np.diff(edges)
    area = 2 * np.pi * ctr * dr
    rho = np.where(area > 0, counts / (area * max(L, 1e-9)), 0.0)
    return ctr, rho, counts

def count_shells_bic(r, kmax=4):
    "Objective wall count: GMM over r, model chosen by minimum BIC."
    distinct = len(np.unique(np.round(r, 3)))
    kmax = min(kmax, distinct)
    if kmax <= 1:
        return 1
    r2 = r.reshape(-1, 1); best, bb = 1, np.inf
    for k in range(1, kmax + 1):
        b = GaussianMixture(k, random_state=0, n_init=1).fit(r2).bic(r2)
        if b < bb: bb, best = b, k
    return best

def nn_distances(cart, cell, axis, k=1):
    "Nearest-neighbour distance per atom, axial periodicity via +/-1 image tiling."
    imgs = np.vstack([cart + s * cell[axis] for s in (-1, 0, 1)])
    d, _ = cKDTree(imgs).query(cart, k=k + 1)
    return d[:, 1:]

def coordination(cart, cell, axis, scale=1.3):
    imgs = np.vstack([cart + s * cell[axis] for s in (-1, 0, 1)])
    tree = cKDTree(imgs)
    nn = np.median(nn_distances(cart, cell, axis))
    cut = scale * nn
    return np.array([len(tree.query_ball_point(p, cut)) - 1 for p in cart]), nn

def ellipticity(u, v):
    cov = np.cov(np.vstack([u, v]))
    w = np.clip(np.linalg.eigvalsh(cov), 1e-9, None)
    return float(np.sqrt(1 - w[0] / w[1])), w

print("Geometry helpers ready.")

## 3 · Six random unit-cell examples

In [ ]:
# --- 6 random unit-cell examples: detailed per-tube metrics -------------------
# `tube_metrics` is the single per-tube descriptor reused by the quality filter.
def tube_metrics(t):
    "Per-tube geometry/chemistry descriptor from cart + cell + elements + frac."
    cart = np.asarray(t["cart"], float); cell = np.asarray(t["cell"], float)
    els  = np.asarray(t["elements"]); n = len(cart)
    ax   = detect_tube_axis(t["frac"])
    L    = float(np.linalg.norm(cell[ax]))
    r, th, z, u, v = cylindrical_coords(cart, cell, ax)
    r05, r95 = (np.percentile(r, [5, 95]) if r.size else (0.0, 0.0))  # mask-matching edges
    ctr, rho, cnt = radial_density(r, L, nbins=24)
    pos = rho[rho > 0]
    rho_peak = float(rho.max()) if rho.size else 0.0
    rho_mean = float(pos.mean()) if pos.size else 0.0
    peak_ratio = (rho_peak / rho_mean) if rho_mean > 0 else 0.0
    walls = max(1, len(find_peaks(np.concatenate([[0], rho, [0]]))[0]))
    nn = nn_distances(cart, cell, ax).ravel() if n > 1 else np.array([])
    cn, _ = coordination(cart, cell, ax) if n > 1 else (np.array([]), None)
    eps = ellipticity(u, v)[0] if n >= 3 else np.nan
    return dict(
        formula=t.get("formula"), nsites=int(n), axis="abc"[ax], L=L,
        r_min=float(r05), r_max=float(r95),
        r_min_abs=float(r.min()) if r.size else np.nan,
        r_max_abs=float(r.max()) if r.size else np.nan,
        wall_thickness=float(r95 - r05), wall_std=float(r.std()) if r.size else np.nan,
        walls=int(walls), peak_ratio=float(peak_ratio),
        min_nn=float(nn.min()) if nn.size else np.nan,
        median_bond=float(np.median(nn)) if nn.size else np.nan,
        mean_cn=float(np.mean(cn)) if len(cn) else np.nan,
        ellipticity=float(eps) if eps == eps else np.nan,
        lin_dens=n / L if L > 0 else np.nan,
    )

_rngA = np.random.default_rng(SEED + 1)
_pick = _rngA.choice(len(templates), size=min(6, len(templates)), replace=False)
examples = [templates[i] for i in _pick]

print(f"6 random unit-cell examples (seed={SEED + 1})\n" + "=" * 74)
for t in examples:
    m = tube_metrics(t)
    raw = f"  [raw n={t['nsites_raw']}, {t.get('reduce_tag', '')}]" if "nsites_raw" in t else ""
    print(f"\n─ {m['formula']}  (n={m['nsites']}, axis={m['axis']}, L={m['L']:.2f} Å){raw}")
    print(f"   radius   r_min(5%)={m['r_min']:.2f}  r_max(95%)={m['r_max']:.2f} Å"
          f"   (abs {m['r_min_abs']:.2f}–{m['r_max_abs']:.2f})")
    print(f"   wall     thickness={m['wall_thickness']:.2f} Å  std={m['wall_std']:.2f} Å  "
          f"walls≈{m['walls']}  ρ_peak/ρ_mean={m['peak_ratio']:.2f}")
    print(f"   bonding  min contact={m['min_nn']:.2f} Å  median bond={m['median_bond']:.2f} Å  "
          f"⟨CN⟩={m['mean_cn']:.2f}")
    print(f"   shape    ellipticity={m['ellipticity']:.2f}  linear density={m['lin_dens']:.2f} atoms/Å")

# cross-section small-multiples (theta, r) coloured by element
fig, axes = plt.subplots(2, 3, figsize=(12, 8), subplot_kw=dict(projection="polar"))
for a, t in zip(axes.flat, examples):
    ax_ = detect_tube_axis(t["frac"])
    r, th, z, u, v = cylindrical_coords(np.asarray(t["cart"], float),
                                        np.asarray(t["cell"], float), ax_)
    els = np.asarray(t["elements"]); cmap = elem_colors(els)
    for el in sorted(set(els)):
        mm = els == el
        a.scatter(th[mm], r[mm], s=20, color=cmap[el], label=str(el),
                  edgecolor="white", linewidth=0.3)
    a.set_title(f"{t.get('formula')}  (n={len(t['cart'])})", fontsize=9, pad=8)
    a.set_xticklabels([]); a.set_yticklabels([])
    a.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.30, 1.15),
             handletextpad=0.1, borderpad=0.1, framealpha=0.6)
for k in range(len(examples), 6):
    axes.flat[k].axis("off")
fig.suptitle("6 random unit-cell examples — cross-section (θ, r) by element",
             y=1.01, fontweight="bold")
plt.tight_layout(); plt.show()


### 3D perspective views of the 6 example tubes

Same 6 tubes as above, in perspective. Each panel marks **r_min** (blue cylinder), **r_max** (red cylinder) about the detected tube axis, and the atoms' **x / y / z extents in Å** on the bounding box.

In [ ]:
# --- 3D perspective views of the 6 example tubes -----------------------------
# Reuses `examples` from the cell above. r_min/r_max cylinders follow the tube
# axis; x/y/z labels give the Cartesian bounding-box extent in Ångström.
import itertools
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d projection)

def _tube_frame(t):
    "Return (cart, elements, a_hat, e1, e2, ctr, r, z) in the tube-axis frame."
    cart = np.asarray(t["cart"], float); cell = np.asarray(t["cell"], float)
    els  = np.asarray(t["elements"])
    ax_i = detect_tube_axis(t["frac"])
    a_hat = cell[ax_i] / np.linalg.norm(cell[ax_i])
    z = cart @ a_hat
    perp = cart - np.outer(z, a_hat)
    other = [k for k in range(3) if k != ax_i]
    e1 = cell[other[0]] - (cell[other[0]] @ a_hat) * a_hat
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(a_hat, e1)
    ctr = perp.mean(0)
    r = np.hypot((perp - ctr) @ e1, (perp - ctr) @ e2)
    return cart, els, a_hat, e1, e2, ctr, r, z

def draw_tube_3d(ax, t):
    cart, els, a_hat, e1, e2, ctr, r, z = _tube_frame(t)
    r_min, r_max = (np.percentile(r, [5, 95]) if r.size else (0.0, 0.0))
    z0, z1 = (z.min(), z.max()) if z.size else (0.0, 1.0)
    cmap = elem_colors(els)
    for el in sorted(set(els)):                       # atoms, coloured by element
        m = els == el
        ax.scatter(cart[m, 0], cart[m, 1], cart[m, 2], s=45, color=cmap[el],
                   edgecolors="k", linewidths=0.3, label=str(el), depthshade=True)
    th = np.linspace(0, 2 * np.pi, 48)                # r_min / r_max cylinders + end rings
    TZ = np.array([z0, z1])
    for R, col in [(r_min, "#277DA1"), (r_max, "#F94144")]:
        TT, ZZ = np.meshgrid(th, TZ)
        S = (ctr[None, None, :] + ZZ[..., None] * a_hat[None, None, :]
             + R * (np.cos(TT)[..., None] * e1[None, None, :]
                    + np.sin(TT)[..., None] * e2[None, None, :]))
        ax.plot_surface(S[..., 0], S[..., 1], S[..., 2], color=col, alpha=0.12,
                        linewidth=0, shade=False)
        ring = ctr + z1 * a_hat + R * (np.cos(th)[:, None] * e1 + np.sin(th)[:, None] * e2)
        ax.plot(ring[:, 0], ring[:, 1], ring[:, 2], color=col, lw=1.6)
    lo, hi = cart.min(0), cart.max(0); d = hi - lo    # bounding box + x/y/z Å labels
    corners = np.array(list(itertools.product(*zip(lo, hi))))
    for i, j in itertools.combinations(range(len(corners)), 2):
        if int((corners[i] != corners[j]).sum()) == 1:
            seg = np.vstack([corners[i], corners[j]])
            ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.6", lw=0.5, alpha=0.5)
    ax.text((lo[0] + hi[0]) / 2, lo[1], lo[2], f"x = {d[0]:.1f} Å", fontsize=8, color="k")
    ax.text(hi[0], (lo[1] + hi[1]) / 2, lo[2], f"y = {d[1]:.1f} Å", fontsize=8, color="k")
    ax.text(hi[0], lo[1], (lo[2] + hi[2]) / 2, f"z = {d[2]:.1f} Å", fontsize=8, color="k")
    try:
        ax.set_box_aspect(tuple(np.maximum(d, 1e-3)))
    except Exception:
        pass
    ax.view_init(elev=18, azim=-60)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(f"{t.get('formula')}\n"
                 f"r$_{{min}}$={r_min:.2f}  r$_{{max}}$={r_max:.2f} Å", fontsize=9)
    ax.legend(fontsize=6, loc="upper left", handletextpad=0.1, borderpad=0.1,
              framealpha=0.6)

fig = plt.figure(figsize=(15, 9.5))
for k, t in enumerate(examples):
    ax = fig.add_subplot(2, 3, k + 1, projection="3d")
    draw_tube_3d(ax, t)
fig.suptitle("3D perspective — atoms, r$_{min}$ (blue) / r$_{max}$ (red) cylinders, "
             "and x/y/z extents (Å)", y=0.99, fontweight="bold")
plt.tight_layout(); plt.show()


## 4 · Quality filter — better templates for the generation mask

Population pass over **all** reduced tubes; the funnel reports how many survive each gate. For this synthetic set the contact gate is expected to dominate (raw candidates carry sub-Ångström overlaps).

In [ ]:
# --- Quality filter: pick better templates for the generation mask ------------
# Gates select templates that give the `shl` mask a physical, guidance-friendly
# geometry. Tune the constants below; the funnel shows how many survive each gate.
MIN_CONTACT = 0.7     # Å  minimum nearest-neighbour distance (reject unphysical overlaps)
MIN_RMIN    = 1.00    # Å  inner hollow radius (r_min, 5th pct); reject filled / non-nanotube cross-sections
NATM_MIN    = 4       # atoms  lower bound (need a real 2D cross-section)
NATM_MAX    = 128     # atoms  DB reduction cap
PEAK_RATIO  = 2.0     # ρ_peak/ρ_mean (density-guidance viability; flat ρ -> no gradient)
SINGLE_WALL = False   # if True, also require walls == 1

# Compute metrics once over the whole population (reused if already built above).
if "M" not in globals() or len(M) != len(templates):
    _t0 = time.time()
    M = pd.DataFrame([tube_metrics(t) for t in templates])
    print(f"Computed metrics for {len(M):,} templates in {time.time() - _t0:.1f}s")

gates = [
    (f"contacts   (min_nn >= {MIN_CONTACT:.2f} Å)", M.min_nn >= MIN_CONTACT),
    (f"hollow     (r_min >= {MIN_RMIN:.2f} Å)", M.r_min >= MIN_RMIN),
    (f"atoms      ({NATM_MIN} <= n <= {NATM_MAX})", (M.nsites >= NATM_MIN) & (M.nsites <= NATM_MAX)),
    (f"peaked ρ   (ρ_peak/ρ_mean >= {PEAK_RATIO:.1f})", M.peak_ratio >= PEAK_RATIO),
]
if SINGLE_WALL:
    gates.append(("single-wall (walls == 1)", M.walls == 1))

keep = np.ones(len(M), bool)
print(f"\n{'gate':42s} {'pass':>8s} {'cumulative':>11s}")
print("-" * 63)
for name, mask in gates:
    mask = mask.fillna(False).values
    keep &= mask
    print(f"{name:42s} {int(mask.sum()):8d} {int(keep.sum()):11d}")
print("-" * 63)
n_keep = int(keep.sum())
print(f"SURVIVING TEMPLATES: {n_keep:,} / {len(M):,}  ({100 * n_keep / max(len(M), 1):.1f}%)")

removed = {name: int((~mask.fillna(False)).sum()) for name, mask in gates}
worst = max(removed, key=removed.get)
print(f"Most-culling gate (standalone): '{worst.strip()}' removes {removed[worst]:,} templates")

M["passes"] = keep
_cols = ["r_min", "r_max", "wall_thickness", "peak_ratio", "min_nn", "nsites"]
print("\nSummary of surviving templates:")
display(M[keep][_cols].describe().round(2))
